# MSCS 634 - Advanced Big Data and Data Mining
## Lab 3: K-Means and K-Medoids Clustering Using the Wine Dataset

**Student:** Vineela Reddy Karnatakam  
**Course:** MSCS 634 - Advanced Big Data and Data Mining  
**Assignment:** Lab 3 - Clustering with K-Means and K-Medoids


## Step 1: Load and Prepare the Dataset

This section loads the Wine dataset from `sklearn`, explores its features and class distribution, and standardizes the feature values using z-score normalization.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import kmedoids

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score, pairwise_distances
from sklearn.decomposition import PCA

wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = pd.Series(wine.target, name='target')

print('Dataset shape:', X.shape)
display(X.head())


In [ ]:
print('Feature names:')
for feature in wine.feature_names:
    print('-', feature)

print('\nClass distribution:')
print(y.value_counts().sort_index())

class_distribution = pd.DataFrame({
    'Class': np.arange(len(wine.target_names)),
    'Class Name': wine.target_names,
    'Count': y.value_counts().sort_index().values
})

display(class_distribution)


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled_df = pd.DataFrame(X_scaled, columns=wine.feature_names)

display(X_scaled_df.head())

print('Approximate standardized means:')
print(np.round(X_scaled_df.mean(), 4))

print('\nApproximate standardized standard deviations:')
print(np.round(X_scaled_df.std(ddof=0), 4))


## Step 2: Implement K-Means Clustering

K-Means is applied with `k = 3`, matching the three known classes in the Wine dataset.


In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)

kmeans_silhouette = silhouette_score(X_scaled, kmeans_labels)
kmeans_ari = adjusted_rand_score(y, kmeans_labels)

print(f'K-Means Silhouette Score: {kmeans_silhouette:.4f}')
print(f'K-Means Adjusted Rand Index: {kmeans_ari:.4f}')


## Step 3: Implement K-Medoids Clustering

K-Medoids is implemented using the `kmedoids` package. A Euclidean pairwise distance matrix is created first, and the FasterPAM algorithm is used with `k = 3`.


In [ ]:
distance_matrix = pairwise_distances(X_scaled, metric='euclidean')

kmedoids_result = kmedoids.fasterpam(distance_matrix, 3)

kmedoids_labels = np.array(kmedoids_result.labels)
medoid_indices = np.array(kmedoids_result.medoids)
kmedoids_centers = X_scaled[medoid_indices]

kmedoids_silhouette = silhouette_score(X_scaled, kmedoids_labels)
kmedoids_ari = adjusted_rand_score(y, kmedoids_labels)

print(f'K-Medoids Silhouette Score: {kmedoids_silhouette:.4f}')
print(f'K-Medoids Adjusted Rand Index: {kmedoids_ari:.4f}')
print('Medoid indices:', medoid_indices)


## Step 4: Visualize and Compare Results

Principal Component Analysis (PCA) is used to reduce the standardized feature set to two dimensions for visualization.


In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

kmeans_centers_pca = pca.transform(kmeans.cluster_centers_)
kmedoids_centers_pca = pca.transform(kmedoids_centers)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=kmeans_labels,
    cmap='viridis',
    s=45,
    alpha=0.8
)

axes[0].scatter(
    kmeans_centers_pca[:, 0],
    kmeans_centers_pca[:, 1],
    c='red',
    marker='X',
    s=220,
    edgecolor='black',
    label='Centroids'
)

axes[0].set_title('K-Means Clustering')
axes[0].set_xlabel('Principal Component 1')
axes[0].set_ylabel('Principal Component 2')
axes[0].legend()

axes[1].scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=kmedoids_labels,
    cmap='viridis',
    s=45,
    alpha=0.8
)

axes[1].scatter(
    kmedoids_centers_pca[:, 0],
    kmedoids_centers_pca[:, 1],
    c='red',
    marker='D',
    s=160,
    edgecolor='black',
    label='Medoids'
)

axes[1].set_title('K-Medoids Clustering')
axes[1].set_xlabel('Principal Component 1')
axes[1].set_ylabel('Principal Component 2')
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
results = pd.DataFrame({
    'Algorithm': ['K-Means', 'K-Medoids'],
    'Silhouette Score': [kmeans_silhouette, kmedoids_silhouette],
    'Adjusted Rand Index (ARI)': [kmeans_ari, kmedoids_ari]
})

display(results.round(4))


## Brief Analysis

The algorithm with the higher **Silhouette Score** produced more clearly separated and internally cohesive clusters. The algorithm with the higher **Adjusted Rand Index (ARI)** produced clusters that agreed more closely with the actual Wine class labels.

K-Means tends to form compact and roughly spherical clusters because it minimizes squared distance to cluster centroids. K-Medoids is similar, but it chooses actual observations as cluster representatives rather than computed mean points.

Because K-Medoids uses actual observations as medoids, it is typically more robust to outliers. K-Means is usually faster and works especially well with large numeric datasets where the clusters are compact and reasonably well separated.

After running the notebook, compare the metric values shown in the results table to determine which approach performed better on this dataset.


## Conclusion

This lab demonstrated the use of K-Means and K-Medoids clustering on the standardized Wine dataset. Silhouette Score and ARI were used to evaluate clustering quality, while PCA provided a two-dimensional visualization of the resulting cluster structures.
